In [1]:
#import libraries
import glob
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
# find and load the parquet data
files = sorted(glob.glob("../../doha/doha_parsed_cases/*.parquet"))
print(files)

['../../doha/doha_parsed_cases/all_cases_1.parquet', '../../doha/doha_parsed_cases/all_cases_2.parquet']


In [3]:
# load the data
df = pd.read_parquet(files[0])
print(df.shape)
print(df.columns)

(17742, 18)
Index(['case_number', 'date', 'outcome', 'guidelines', 'summary', 'full_text',
       'sor_allegations', 'mitigating_factors', 'judge', 'source_url',
       'formal_findings', 'case_type', 'appeal_board_members', 'who_appealed',
       'judges_findings_of_fact', 'judges_analysis', 'discussion', 'order'],
      dtype='str')


In [5]:
# clean the dataset (removes incomplete cases, randomizes the cases)and select 2000 cases
df = df.dropna(subset=['full_text', 'outcome']).sample(frac=1, random_state=42)
df = df.head(2000)
print(df.shape)
print(df['outcome'].value_counts())

(2000, 18)
outcome
DENIED      1163
GRANTED      715
UNKNOWN      113
REMANDED       9
Name: count, dtype: int64


In [8]:
# split the text into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    df['full_text'], 
    df['outcome'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['outcome']
)

print("Training cases:", len(X_train))
print("Validation cases:", len(X_val))

Training cases: 1600
Validation cases: 400


In [9]:
# create a TF-IDF vectorizer
vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))

Xtr = vec.fit_transform(X_train) #x training data
Xv = vec.transform(X_val)   # x validation data

print("Training TF-IDF shape:", Xtr.shape)
print("Validation TF-IDF shape:", Xv.shape)

Training TF-IDF shape: (1600, 2000)
Validation TF-IDF shape: (400, 2000)
